## Table of Contents
1. [Importing Libraries](#Importing-Libraries)
2. [Loading Data](#Loading-Data)
3. [Feature Engineering](#Feature-Engineering)
4. [Model](#Model)

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

import gc
import warnings
warnings.simplefilter('ignore')

### Loading Data

In [ ]:
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")
orig = pd.read_csv("/kaggle/input/datasets/cdeotte/s6e4-original-dataset/Heart_Disease_Prediction.csv")

# print shape
print(train.shape)
print(test.shape)
print(orig.shape)

target = 'Heart Disease'
cat_cols = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 
                    'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
num_cols = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
cols = [col for col in train.columns if col != ['id', target]]

# Encoding the target column to numeric
train['Heart Disease']=pd.get_dummies(train['Heart Disease'],drop_first=True,dtype=int)
orig['Heart Disease']=pd.get_dummies(orig['Heart Disease'],drop_first=True,dtype=int)

### Feature Engineering

In [ ]:
from itertools import combinations
def create_features(train_df, test_df, orig_df, num_cols, cat_cols, target_col):
    
    # 1. Frequency Encoding (Train + Test + Orig)
    for col in num_cols:
        combined = pd.concat([train_df[col], test_df[col], orig_df[col]], axis=0)
        fe_map = combined.value_counts().to_dict()
        
        new_col = f'FE_{col}'
        train_df[new_col] = train_df[col].map(fe_map)
        test_df[new_col] = test_df[col].map(fe_map)
        
    # 2. Target Encoding (Train + Orig)
    train_orig_df = pd.concat([train_df, orig_df], axis=0).reset_index(drop=True)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    global_mean = train_orig_df[target_col].mean()

    for col in num_cols:
        new_col_name = f'TE_{col}'
        train_df[new_col_name] = 0.0

        for train_idx, val_idx in skf.split(train_df, train_df[target_col]):
            fold_train = train_df.iloc[train_idx]
            combined_source = pd.concat([fold_train, orig_df], axis=0)
            
            mean_target = combined_source.groupby(col)[target_col].mean()
            train_df.loc[val_idx, new_col_name] = train_df.loc[val_idx, col].map(mean_target)
            
        train_df[new_col_name] = train_df[new_col_name].fillna(global_mean)
        test_mean = train_orig_df.groupby(col)[target_col].mean()
        test_df[new_col_name] = test_df[col].map(test_mean).fillna(global_mean)

    # 3. Numeric Binning
    for df in [train_df, test_df]:
        for col in num_cols:
            df[f'bin_{col}'] = pd.qcut(df[col], q=10, labels=False, duplicates='drop')

# Call the function
create_features(train, test, orig, num_cols, cat_cols, target)

In [ ]:
### sanity check
display(train.info(max_cols=200))
print('='*40)
display(test.info(max_cols=200))

In [ ]:
### preparing the data
X_train = train.drop([target, 'id'], axis=1) 
y_train = train[target]
X_test = test.drop(['id'], axis=1)

# print shape
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")
print(f"X_test : {X_test.shape}")

### Model

In [ ]:
# Params
RANDOM_STATE = 42
N_SPLITS     = 5

params_cat = {
    'objective'         : 'Logloss',
    'eval_metric'       : 'AUC',
    'iterations'        : 2000,
    'learning_rate'     : 0.05,
    'depth'             : 5,              # Similar to num_leaves=31
    'min_data_in_leaf'  : 20,             # Similar to min_child_samples
    'subsample'         : 0.8,
    'bootstrap_type'    : 'Bernoulli',
    'colsample_bylevel' : 0.8,
    'reg_lambda'        : 0.0,            # L2 regularization
    'random_state'      : RANDOM_STATE,
    'verbose'           : False,
    'early_stopping_rounds': 50,
    'thread_count'         : -1, 
    'metric_period'      : 1
}

# SKF
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Training
oof_preds_cat = np.zeros(len(X_train))
test_preds_cat = np.zeros(len(X_test))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\nStarting Fold {fold + 1} of {N_SPLITS}...")
    print("-" * 30)
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx],  y_train.iloc[val_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
    val_pool   = Pool(X_val, y_val, cat_features=cat_cols)
    
    model_cat = CatBoostClassifier(**params_cat)
    
    model_cat.fit(
        train_pool,
        eval_set=val_pool,
        verbose_eval=200,
        use_best_model=True
    )
    
    # OOF Predictions
    val_preds = model_cat.predict_proba(X_val)[:, 1]
    oof_preds_cat[val_idx] = val_preds

    # test predictions
    fold_test_preds_cat = model_cat.predict_proba(X_test)[:, 1]
    test_preds_cat += fold_test_preds_cat / N_SPLITS
    
    val_auc = roc_auc_score(y_val, val_preds)
    fold_scores.append(val_auc)
    print(f"    Val AUC: {val_auc:.4f}")

    # cleanup
    del model_cat
    gc.collect()
    
# Final CV Results
oof_auc_cat = roc_auc_score(y_train, oof_preds_cat)
mean_auc = np.mean(fold_scores)
std_auc = np.std(fold_scores)

print("\n" + "="*60)
print(f"OOF AUC: {oof_auc_cat:.6f} | Mean AUC: {mean_auc:.6f} | Std AUC: {std_auc:.4f}")
print("="*60)

In [ ]:
### creating submission file

submission = pd.DataFrame({
    'id': test['id'],
    'Heart Disease': test_preds_cat
})

submission.to_csv('submission.csv', index=False)

print(f"  Submission shape: {submission.shape}")
print(f"\nFirst 5 rows:")
print(submission.head())